前言：第七篇，给 Agent 一本菜谱，而不是一桌满汉全席
本篇干一件事：解决知识把上下文撑爆的问题。

先说场景。你的项目里多半有这些东西——一份 React 组件规范、一份 SQL 风格指南、一份 API 设计文档。你希望 Agent 干活的时候遵守这些规矩，最直接的做法是什么？全部拼进 system prompt：

SYSTEM = (
    f"You are a coding agent. "
    + open("docs/react-style.md").read()
    + open("docs/sql-style.md").read()
    + open("docs/api-design.md").read()
)
能跑，Agent 也确实读到了所有规范。但这个写法埋着一个问题，上一篇咱们刚算过一笔类似的账——它和分身要解决的是同一本账上的两页：上下文里到底该装什么。

上一篇解决的是"过程去哪"：调查类的工具调用派给分身，白纸上烧掉，只带结论回来。这一篇解决"知识去哪"：那些成套的规范文档，不该常驻，该用到才加载。

读完你会拿到三样东西：

SkillLoader 的完整实现
：扫描技能目录、解析元信息、拼一张只印"菜名"的菜谱，总共四十来行
"目录常驻、正文按需"的两层注入机制
：名称和描述进 system prompt，全文走一个叫 load_skill 的工具按需回流
三个最容易想错的点：为什么做成工具而不是让模型自己去读文件；加载进来的内容到底进了哪（不是 system prompt）；以及技能系统最常见的死法——description 写歪了，模型永远想不起来加载
门槛不变：会 Python 基础语法、手上有一份能跑的 Agent 循环（工具分发、权限、hooks、清单、分身都在的那种）。直接开始。

PART 01：案发现场——system prompt 是个每轮都要重发的包裹
先把 system prompt 的机制账算清楚，你才知道那个拼字符串的写法亏在哪。

很多人对 system prompt 的理解停留在"开机设置"：启动的时候设置一次，之后就存在那了。这个直觉是错的，而且错得很贵。

LLM API 是无状态的。你眼里的一场连续对话，在模型那边是几十次独立的调用：每轮都把完整的历史重新发一遍——messages[] 是历史，system 也是历史的一部分。system prompt 不是"设置一次"，是"每一轮都全文重发一次"。

这带来一个直接后果：塞进 system prompt 的每一个字，不是存一次的仓储费，是每一轮都要重付的过路费。

来算笔具体的账。假设那三份文档加起来 1.5 万 token（规范文档写得细一点，这个数只多不少）。现在你的任务只是改一个 React 组件——真正需要的只有 React 规范那一份，SQL 风格指南和 API 设计文档全程用不上。

但它们在 system prompt 里。意味着：

第 1 轮调用，发 1.5 万 token 的文档
第 10 轮调用，还是发 1.5 万 token 的文档
这个任务干完 20 轮，那两份无关文档被重发了 20 次，白付 30 万 token 的输入
而且钱只是小事。更贵的是注意力：上下文窗口是有限的，被无关文档占掉的空间，本来可以放代码、放对话、放工具结果。窗口快满的时候，模型的表现会肉眼可见地往下掉——它要在一堆和当前任务无关的文字里，翻找真正相关的那部分。